# Chess Elo Predictor (3-Class XGBoost)


In [1]:
import pandas as pd
import numpy as np
import chess.pgn
import xgboost as xgb
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

## PART 1: Feature Extraction

In [2]:
features_file = 'ultimate_features_full.csv'

def parse_scores(score_str):
    if pd.isna(score_str) or score_str == '': return []
    scores = []
    for s in str(score_str).split():
        if s.upper() == 'NA': scores.append(np.nan)
        else:
            try: scores.append(int(s))
            except: scores.append(np.nan)
    return scores

def get_cpl_and_chaos(scores, color):
    valid_scores = [np.clip(s, -1500, 1500) for s in scores if not pd.isna(s)]
    if len(valid_scores) < 2:
        return {
            f'{color}_avg_cpl': 0,
            f'{color}_max_swing': 0,
            f'{color}_chaos': 0,
            f'{color}_blunders': 0
        }

    losses, swings, blunders = [], [], 0
    start = 0 if color == 'white' else 1

    for i in range(start, len(valid_scores)-2, 2):
        diff = valid_scores[i] - valid_scores[i+2] if color == 'white' else valid_scores[i+2] - valid_scores[i]
        if diff > 0:
            losses.append(diff)
            if diff > 300: blunders += 1
        else:
            losses.append(0)
        swings.append(abs(valid_scores[i] - valid_scores[i+2]))

    return {
        f'{color}_avg_cpl': np.mean(losses) if losses else 0,
        f'{color}_max_swing': max(swings) if swings else 0,
        f'{color}_chaos': sum(swings) if swings else 0,
        f'{color}_blunders': blunders
    }

In [3]:
if not os.path.exists(features_file):
    print("Building dataset...")
    stockfish_df = pd.read_csv('stockfish.csv')
    stockfish_dict = dict(zip(stockfish_df['Event'], stockfish_df['MoveScores']))

    games_data = []
    with open('data.pgn', 'r') as f:
        for _ in tqdm(range(25000)):
            game = chess.pgn.read_game(f)
            if game is None: break

            w_elo = game.headers.get('WhiteElo', '?')
            b_elo = game.headers.get('BlackElo', '?')
            if w_elo == '?' or b_elo == '?': continue

            event_id = int(game.headers.get('Event', 0))
            moves = list(game.mainline_moves())
            if len(moves) < 10: continue

            row = {'Event': event_id, 'WhiteElo': int(w_elo), 'BlackElo': int(b_elo), 'NumMoves': len(moves)}

            board = chess.Board()
            w_caps = b_caps = w_checks = b_checks = 0

            for move in moves:
                if board.is_capture(move):
                    if board.turn == chess.WHITE: w_caps += 1
                    else: b_caps += 1
                board.push(move)

            row.update({'W_Caps': w_caps, 'B_Caps': b_caps})
            scores = parse_scores(stockfish_dict.get(event_id, ''))
            row.update(get_cpl_and_chaos(scores, 'white'))
            row.update(get_cpl_and_chaos(scores, 'black'))

            games_data.append(row)

    df = pd.DataFrame(games_data).fillna(0)
    df.to_csv(features_file, index=False)
else:
    df = pd.read_csv(features_file)

df.head()

Building dataset...


100%|██████████| 25000/25000 [04:38<00:00, 89.75it/s] 


,Event,WhiteElo,BlackElo,NumMoves,W_Caps,B_Caps,white_avg_cpl,white_max_swing,white_chaos,white_blunders,black_avg_cpl,black_max_swing,black_chaos,black_blunders
0,1,2354,2411,38,4,5,6.833333,74,263,0,7.555556,30,235,0
1,2,2523,2460,13,1,1,2.000000,22,53,0,7.400000,26,66,0
2,3,1915,1999,106,11,12,36.115385,977,2230,1,12.807692,1006,2883,0
3,4,2446,2191,77,8,9,6.736842,64,706,0,11.540541,68,693,0
4,5,2168,2075,49,8,5,4.375000,294,1059,0,45.652174,286,1296,0


## PART 2: Data Preparation

In [4]:
def get_bracket(elo):
    if elo < 1800: return 0
    elif elo < 2200: return 1
    else: return 2

df['W_Class'] = df['WhiteElo'].apply(get_bracket)

df['W_Volatility'] = df['white_blunders'] / (df['NumMoves'] + 1)
df['B_Volatility'] = df['black_blunders'] / (df['NumMoves'] + 1)

feature_cols = [c for c in df.columns if c not in ['Event', 'WhiteElo', 'BlackElo', 'W_Class']]
X = df[feature_cols].values
y = df['W_Class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

## PART 3: Model Training

In [5]:
model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    n_jobs=-1,
    early_stopping_rounds=50
)

model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=100
)

[0]	validation_0-mlogloss:1.09235
[100]	validation_0-mlogloss:0.96844
[200]	validation_0-mlogloss:0.94971
[300]	validation_0-mlogloss:0.93375
[400]	validation_0-mlogloss:0.92127
[500]	validation_0-mlogloss:0.91073
[600]	validation_0-mlogloss:0.90238
[700]	validation_0-mlogloss:0.89429
[800]	validation_0-mlogloss:0.88987
[900]	validation_0-mlogloss:0.88563
[999]	validation_0-mlogloss:0.88216


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from

## PART 4: Evaluation

In [6]:
predictions = model.predict(X_test)

acc = accuracy_score(y_test, predictions)
print(f"Accuracy: {acc * 100:.2f}%")

print(classification_report(y_test, predictions))

cm = confusion_matrix(y_test, predictions)
pd.DataFrame(cm)

Accuracy: 56.66%
              precision    recall  f1-score   support

           0       0.15      0.17      0.16       307
           1       0.42      0.46      0.44      1661
           2       0.71      0.66      0.69      3021

    accuracy                           0.57      4989
   macro avg       0.43      0.43      0.43      4989
weighted avg       0.58      0.57      0.57      4989



,0,1,2
0,52,174,81
1,165,767,729
2,133,880,2008
